Import necessary libraries

In [1]:
from langchain_community.document_loaders import PyPDFLoader
import json
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

/tmp/ipykernel_483673/137333164.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Load required documents

In [2]:
loader = PyPDFLoader("govt_law_chpt_10.pdf")
pdf_pages = loader.load()
print(len(pdf_pages))

73


In [3]:
with open("govt_law_chpt_10_structure.json", "r") as f:
    index_data = json.load(f)

Flatten Hierarchical JSON to Langchain Documents while storing metadeta

In [4]:
index_docs = []

In [5]:
def process_node(node, parent=None):

    index_docs.append(
        Document(
            page_content=f"""
            Title: {node.get('title','')}
            Summary: {node.get('summary','')}
            """,
            metadata={
                "node_id": node.get("node_id"),
                "title": node.get("title"),
                "parent": parent,
                "start_page": node.get("start_index"),
                "end_page": node.get("end_index")
            }
        )
    )

    for child in node.get("nodes", []):
        process_node(child, node.get("title"))

In [6]:
for node in index_data["structure"]:
    process_node(node)

print(len(index_docs))

363


BM25 retriever

In [8]:
section_retriever = BM25Retriever.from_documents(index_docs)
section_retriever.k = 5

In [9]:
query = "What is the purpose of the Act?"

sections = section_retriever.invoke(query)

for s in sections:
    print(s.metadata["title"])

The purpose of the Act
Section 1-1. The purpose of the Act
Section 1-4. Undertakings with no employees, etc.
Section 1-5. Work performed at the home of the employee or employer
Section 4-1. General requirements regarding the working environment


In [2]:
from langchain_community.document_loaders import PyPDFLoader


def extract_pdf_pages(pdf_path, start_page, end_page):
    """
    Extract text from given page range of a PDF.

    Args:
        pdf_path (str): Path to PDF file
        start_page (int): Starting page number (1-based)
        end_page (int): Ending page number (1-based)

    Returns:
        str: Extracted text stored in one variable
    """

    # Load PDF
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    total_pages = len(docs)

    # Validate range
    if start_page < 1 or end_page > total_pages or start_page > end_page:
        return "Invalid page range"

    # required pages
    selected_docs = docs[start_page - 1:end_page]

    stored_data = ""

    for doc in selected_docs:
        print(doc.page_content)
        stored_data += doc.page_content

    return stored_data


# Example usage
data = extract_pdf_pages("data/govt_law_chpt_10.pdf", 15, 16)

print("\nStored Data:")
print(data)

SSeeccttiioonn  33--22..Special safety precautions
(1) In order to maintain safety at the workplace, the employer shall ensure:
a) that employees are informed of accident risks and health hazards that may
be connected with the work, and that they receive the necessary training,
practice and instruction,
b) that employees charged with directing or supervising other employees
have the necessary competence to ensure that the work is performed in a
proper manner with regard to health and safety,
c) expert assistance, when this is necessary in order to implement the require-
ments of this Act.
(2) When satisfactory precautions to protect life and health cannot be
achieved by other means, the employer shall ensure that satisfactory per-
sonal protective equipment is made available to the employees, that the
employees are trained in the use of such equipment and that the equip-
ment is used.
(3) If work is to be carried out that may involve particular hazards to life or
health, written instru

In [3]:
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_16348/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/mnt/e/Vectorless RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def extract_pdf_pages(pdf_path, start_page, end_page):
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    total_pages = len(docs)
    if start_page < 1 or end_page > total_pages:
        return ""
    selected_docs = docs[start_page - 1:end_page]
    stored_data = ""
    for doc in selected_docs:
        stored_data += doc.page_content + "\n\n"
    return stored_data

In [5]:
data = extract_pdf_pages("data/govt_law_chpt_10.pdf", 15, 16)

print("\nStored Data:")
print(data)


Stored Data:
SSeeccttiioonn  33--22..Special safety precautions
(1) In order to maintain safety at the workplace, the employer shall ensure:
a) that employees are informed of accident risks and health hazards that may
be connected with the work, and that they receive the necessary training,
practice and instruction,
b) that employees charged with directing or supervising other employees
have the necessary competence to ensure that the work is performed in a
proper manner with regard to health and safety,
c) expert assistance, when this is necessary in order to implement the require-
ments of this Act.
(2) When satisfactory precautions to protect life and health cannot be
achieved by other means, the employer shall ensure that satisfactory per-
sonal protective equipment is made available to the employees, that the
employees are trained in the use of such equipment and that the equip-
ment is used.
(3) If work is to be carried out that may involve particular hazards to life or
health, 